# Data Preprocessing

This notebook prepares the dataset for machine learning by applying a complete text preprocessing pipeline. The workflow includes cleaning the raw text, normalizing its format, removing unnecessary tokens, and generating a processed version suitable for feature extraction and model training.

In [ ]:
import pandas as pd
df = pd.read_parquet("../data/eda.parquet")

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

df.head()

,text,label,label_text,word_count,char_count,utterance_type
0,I am still waiting on my card?,11,card_arrival,7,30,Question
1,What can I do if my card still hasn't arrived after 2 weeks?,11,card_arrival,13,60,Question
2,I have been waiting over a week. Is the card still coming?,11,card_arrival,12,58,Question
3,Can I track my card while it is in the process of delivery?,11,card_arrival,13,59,Question
4,"How do I know if I will get my card, or if it is lost?",11,card_arrival,15,54,Question


## Text Preprocessing Pipeline

To prepare the text for feature extraction and modeling, a preprocessing pipeline was applied to each customer message.

The pipeline performs the following operations:

1. Convert all text to lowercase for consistent word representation.
2. Expand English contractions (e.g., *don't* → *do not*).
3. Remove unwanted characters using regular expressions while preserving alphabetic words.
4. Normalize whitespace by replacing multiple spaces with a single space and trimming leading/trailing spaces.
5. Tokenize the cleaned text into individual words.

The resulting tokens were stored in a new **`tokens`** column, while preserving the original text for future reference.

In [ ]:
from nltk.tokenize import word_tokenize
import contractions
import re

def preprocess_text(text):
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return word_tokenize(text)

df["tokens"] = df["text"].apply(preprocess_text)

In [ ]:
df[["text", "tokens", "utterance_type"]].sample(5, random_state=42)

,text,tokens,utterance_type
6883,Is it possible for me to change my PIN number?,"[is, it, possible, for, me, to, change, my, pin, number]",Question
5836,I'm not sure why my card didn't work,"[i, am, not, sure, why, my, card, did, not, work]",Request
8601,I don't think my top up worked,"[i, do, not, think, my, top, up, worked]",Request
2545,Can you explain why my payment was charged a fee?,"[can, you, explain, why, my, payment, was, charged, a, fee]",Question
8697,"How long does a transfer from a UK account take? I just made one and it doesn't seem to be working, wondering if everything is okay","[how, long, does, a, transfer, from, a, uk, account, take, i, just, made, one, and, it, does, not, seem, to, be, working, wondering, if, everything, is, okay]",Question


## Stop Word Removal

A dedicated preprocessing step was applied to remove common English stop words using the **NLTK** stop word list.

However, the negation terms **`no`**, **`nor`**, and **`not`** were intentionally preserved because they carry important semantic information in customer support messages. Removing these words could change the meaning of a request (e.g., *"card not received"*).

The filtered tokens were stored in a new **`tokens_no_stop`** column while retaining the original tokenized text for comparison.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words.difference_update({'not', 'no', 'nor'})

def remove_stopwords(tokens):
    return [
        word 
        for word in tokens 
        if word not in stop_words
    ]

df["tokens_no_stop"] = df["tokens"].apply(remove_stopwords)

In [ ]:
df[["tokens", "tokens_no_stop", "label_text"]].sample(5, random_state=42)

,tokens,tokens_no_stop,label_text
6883,"[is, it, possible, for, me, to, change, my, pin, number]","[possible, change, pin, number]",change_pin
5836,"[i, am, not, sure, why, my, card, did, not, work]","[not, sure, card, not, work]",declined_card_payment
8601,"[i, do, not, think, my, top, up, worked]","[not, think, top, worked]",top_up_failed
2545,"[can, you, explain, why, my, payment, was, charged, a, fee]","[explain, payment, charged, fee]",card_payment_fee_charged
8697,"[how, long, does, a, transfer, from, a, uk, account, take, i, just, made, one, and, it, does, not, seem, to, be, working, wondering, if, everything, is, okay]","[long, transfer, uk, account, take, made, one, not, seem, working, wondering, everything, okay]",balance_not_updated_after_bank_transfer


## Custom Stop Word Identification

After removing the standard English stop words, the remaining tokens were examined by their frequency to identify common words that might contribute little to intent discrimination.

The most frequent tokens were inspected, and selected high-frequency words were further analyzed based on their distribution across intent labels. This analysis showed that words such as **`please`** and **`help`** appeared across the majority of intents, indicating that they function primarily as generic conversational terms rather than intent-specific features.

The resulting filtered tokens were stored in a new **`tokens_clean`** column for subsequent analysis and modeling.

In [ ]:
df_freq = (
    df["tokens_no_stop"]
    .explode()
    .value_counts()
    .reset_index()
)

df_freq = df_freq.sort_values(by="count", ascending=False)
df_freq.head(30)

,tokens_no_stop,count
0,card,2672
1,not,2659
2,account,1348
3,money,1130
4,transfer,1081
5,get,808
6,payment,746
7,need,698
8,cash,690
9,top,616


In [66]:
analysis = pd.DataFrame({
    "token": ["please", "help"],
    "labels_appeared": [69, 60],
    "coverage_%": [89.6, 77.9],
    "most_common_label": [
        "cancel_transfer",
        "cash_withdrawal_not_received"
    ],
    "frequency": [37, 31],
})

analysis

,token,labels_appeared,coverage_%,most_common_label,frequency
0,please,69,89.6,cancel_transfer,37
1,help,60,77.9,cash_withdrawal_not_received,31


In [ ]:
custom_words = {"please", "help"}

def remove_custom_words(tokens):
    return [token for token in tokens if token not in custom_words]

df["tokens_clean"] = df["tokens_no_stop"].apply(remove_custom_words)

## Empty Token Validation

After removing stop words and custom high-frequency words, the resulting token lists were checked for empty sequences.

This validation ensures that no text samples became empty during preprocessing, preventing potential issues in subsequent feature extraction and model training.

In [ ]:
empty_tokens = df["tokens_clean"].str.len() == 0
print(empty_tokens.sum())

0


## Lemmatization

The final preprocessing step applied **lemmatization** to normalize words while preserving their semantic meaning.

Several lemmatization approaches were evaluated during development. **spaCy** was ultimately selected because it performs **part-of-speech (POS) tagging** before lemmatization, allowing each word to be normalized according to its grammatical role within the text.

This approach produced more reliable lemmas for the Banking77 dataset and helped preserve the intended meaning of domain-specific terms, making it more suitable for subsequent feature extraction and model training.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def lemmatize(tokens):

    text = " ".join(tokens)
    doc = nlp(text)
    
    return [
        token.lemma_
        for token in doc
    ]

df["lemmatized"] = df["tokens_clean"].apply(lemmatize)

In [ ]:
df[["tokens_clean", "lemmatized"]].sample(5, random_state=13)

,tokens_clean,lemmatized
5953,"[made, card, payment, not, work, not]","[make, card, payment, not, work, not]"
1783,"[come, tried, pay, contactless, bus, not, work]","[come, try, pay, contactless, bus, not, work]"
4812,"[payment, showing, app, not, cancel, payment, refund, money]","[payment, show, app, not, cancel, payment, refund, money]"
145,"[not, gotten, new, card]","[not, get, new, card]"
4287,"[possible, friends, top, account]","[possible, friend, top, account]"


## Preparing Text for Feature Extraction

After completing all preprocessing steps, the lemmatized tokens were joined into a single text string and stored in a new **`processed_text`** column.

This step prepares the data for vectorization, as feature extraction techniques such as **TF-IDF** and **CountVectorizer** expect raw text strings rather than token lists.

The resulting `processed_text` column serves as the final input for the modeling stage.

In [ ]:
df["processed_text"] = df["lemmatized"].apply(" ".join)

## Save Preprocessed Dataset

The fully preprocessed dataset was saved in **Parquet** format for use in the modeling stage.

Separating preprocessing from modeling improves workflow organization, avoids repeating preprocessing operations, and ensures that all experiments use the same cleaned and reproducible dataset.

In [ ]:
df.to_parquet("../data/preprocessed.parquet", index=False)